# 05 — Probability Calibration (Phase 6)

This notebook is a thin, human-readable wrapper around `src/calibration/`.
It fits and inspects calibration exactly as `scripts/run_calibration_shap_packaging.py`
does, so results here should match the script's outputs.

**No metrics in this notebook are fabricated.** Cell outputs are empty because
this notebook has not yet been executed in this environment — run it locally
(`jupyter nbconvert --to notebook --execute`) to populate real outputs.

In [ ]:
from src.config import get_default_config
from src.data.load import load_and_validate_data
from src.data.split import train_test_split_data
from src.models.pipeline import build_model_pipeline
from src.calibration.calibrator import fit_calibrated_model_with_split
from src.calibration.evaluate import compare_calibration, plot_calibration_curve, plot_probability_distribution

config = get_default_config()
config.data.target_column = "<set explicitly — dataset-specific>"
config.validate()

In [ ]:
df, schema = load_and_validate_data(config.data)
X_train, X_test, y_train, y_test = train_test_split_data(
    df, config.data.target_column, config.split
)
print(X_train.shape, X_test.shape)

In [ ]:
base_pipeline = build_model_pipeline(config.model, config.preprocessing)

fit_result, X_calib, y_calib = fit_calibrated_model_with_split(
    base_pipeline, X_train, y_train, config.calibration
)

fit_result.strategy, fit_result.method

## Compare pre- vs post-calibration probabilities

Computed on `X_calib`/`y_calib` (held out from `X_train`, never the test set).

In [ ]:
y_proba_before = fit_result.raw_pipeline.predict_proba(X_calib)[:, 1]
y_proba_after = fit_result.calibrated_model.predict_proba(X_calib)[:, 1]

comparison = compare_calibration(y_calib, y_proba_before, y_proba_after)
comparison

In [ ]:
fig1 = plot_calibration_curve(y_calib, y_proba_before, y_proba_after)
fig1

In [ ]:
fig2 = plot_probability_distribution(y_proba_before, y_proba_after)
fig2

## Interpretation notes

- `roc_auc_delta` should be near zero — calibration reshapes probabilities,
  it does not change rank ordering. A large ROC-AUC change here would
  indicate a bug, not an improvement.
- Prefer `brier_score_delta` and `log_loss_delta` (negative = improvement)
  when judging whether calibration helped.
- Do not select a calibration method using the test set — this notebook,
  like the pipeline script, only ever touches `X_calib`/`y_calib`.